In [ ]:
# pre-processing
## from jupyter notebook - select individuals studied
for c in 1 2 3 4 5
do
  echo "=== chr${c}: norm + set-id + plink ==="

  bcftools norm -m -any -Oz \
    -o gt_only.present.chr${c}.norm.vcf.gz \
    gt_only.present.chr${c}.vcf.gz
  bcftools index -t gt_only.present.chr${c}.norm.vcf.gz

  bcftools annotate -Oz \
    -o gt_only.present.chr${c}.norm.id.vcf.gz \
    --set-id '%CHROM:%POS:%REF:%ALT' \
    gt_only.present.chr${c}.norm.vcf.gz
  bcftools index -t gt_only.present.chr${c}.norm.id.vcf.gz

  plink --vcf gt_only.present.chr${c}.norm.id.vcf.gz \
        --double-id \
        --make-bed \
        --out gt_only.present.chr${c}
done


printf "gt_only.present.chr2\ngt_only.present.chr3\ngt_only.present.chr4\ngt_only.present.chr5\n" > merge_list.chr2_5.txt

## Merge

plink --bfile gt_only.present.chr1 \
      --merge-list merge_list.chr2_5.txt \
      --make-bed \
      --out gt_only.present.chr1_5

## QC
plink --bfile gt_only.present.chr1_5 \
      --geno 0.05 \
      --mind 0.05 \
      --maf 0.05 \
      --make-bed \
      --out gt_only.present.chr1_5.qc

## LD Pruning
plink --bfile gt_only.present.chr1_5.qc \
      --indep-pairwise 50 5 0.2 \
      --out gt_only.present.chr1_5.qc

plink --bfile gt_only.present.chr1_5.qc \
      --extract gt_only.present.chr1_5.qc.prune.in \
      --make-bed \
      --out gt_only.present.chr1_5.qcpruned



In [ ]:
# PLINK IBD
plink --bfile gt_only.present.chr1_5.qcpruned \
      --genome full \
      --out plink_genome.present.chr1_5.qcpruned

In [ ]:
# To get runtime and storage cost
mkdir -p bench_logs
(/usr/bin/time -v plink --bfile gt_only.present.chr1_5.qcpruned \
  --genome full \
  --out bench_plink_genome) \
  2>&1 | tee bench_logs/plink_genome.timev.txt

# Elapsed (wall clock) time (h:mm:ss or m:ss): 0:00.17
# Maximum resident set size (kbytes): 86096

In [ ]:
# GERMLINE
for c in 1 2 3 4 5
do
  echo "=== Make PED/MAP for chr${c} ==="
  plink --bfile gt_only.present.chr1_5.qcpruned \
        --chr ${c} \
        --recode 12 \
        --out gt_only.present.chr${c}.qcpruned.nomissSNP_ped
done

In [ ]:
for c in 1 2 3 4 5
do
  echo "=== GERMLINE chr${c} ==="
  germline \
    -input gt_only.present.chr${c}.qcpruned.nomissSNP_ped.ped gt_only.present.chr${c}.qcpruned.nomissSNP_ped.map \
    -output germline.chr${c}.qcpruned \
    -min_m 3 \
    -err_hom 0 \
    -err_het 2 \
    -bits 64
done

In [ ]:
cat germline.chr1.qcpruned.match germline.chr2.qcpruned.match germline.chr3.qcpruned.match germline.chr4.qcpruned.match germline.chr5.qcpruned.match \
  > germline.chr1_5.qcpruned.perchr_merged.match

In [3]:
# To get runtime and storage cost of GERMLINE

In [ ]:
mkdir -p bench_logs

# (A) pre-process: get PED/MAP
for c in 1 2 3 4 5
do
  echo "=== BENCH: PLINK recode12 chr${c} ==="
  (/usr/bin/time -v plink --bfile gt_only.present.chr1_5.qcpruned \
        --chr ${c} \
        --recode 12 \
        --out bench_chr${c}.qcpruned.nomissSNP_ped) \
    2>&1 | tee bench_logs/plink_recode12_chr${c}.timev.txt
done

# (B) GERMLINE
for c in 1 2 3 4 5
do
  echo "=== BENCH: GERMLINE chr${c} ==="
  (/usr/bin/time -v germline \
    -input bench_chr${c}.qcpruned.nomissSNP_ped.ped bench_chr${c}.qcpruned.nomissSNP_ped.map \
    -output bench_germline_chr${c} \
    -min_m 3 \
    -err_hom 0 \
    -err_het 2 \
    -bits 64) \
    2>&1 | tee bench_logs/germline_chr${c}.timev.txt
done

In [ ]:
# Runtime and storage cost Summary
cat bench_logs/summary.tsv
tool    chr     elapsed max_rss_kb
plink_recode12  chr1    0:00.09 30152
germline        chr1    0:00.43 16148
plink_recode12  chr2    0:00.09 29120
germline        chr2    0:00.41 15932
plink_recode12  chr3    0:00.08 26688
germline        chr3    0:00.37 14204
plink_recode12  chr4    0:00.08 26912
germline        chr4    0:00.36 14628
plink_recode12  chr5    0:00.07 24712
germline        chr5    0:00.36 12328